# Contenido

1. Antes de calcular: ¿qué quieres conseguir?
2. Factores que determinan el tamaño
3. Estimar una proporción
4. Corrección para una población finita
5. Estimar una media
6. Estudio piloto
7. Potencia estadística
8. Simulación
9. Curvas de aprendizaje en machine learning
10. Ajustes de un estudio real
11. Representatividad y sesgos
12. Censo y reglas rápidas
13. Guía de decisión y función práctica
14. Ejercicios, soluciones y checklist final

# ESTADÍSTICA II: CÓMO DETERMINAR EL TAMAÑO DE UNA MUESTRA

#### Cuaderno complementario Codex

Este cuaderno completa el apartado **Muestreo** del curso de Estadística Inferencial. Su objetivo es que puedas decidir, antes de recoger los datos, cuántas observaciones necesitas y por qué.

Al terminar podrás distinguir los métodos basados en **precisión**, **potencia estadística**, **simulación** y **curvas de aprendizaje**, e incorporar ajustes realistas como población finita, no respuesta, abandono y muestreo por conglomerados.

## Objetivos de aprendizaje

1. Comprender por qué no existe una única fórmula universal.
2. Elegir el método según la pregunta y el tipo de variable.
3. Calcular una muestra para estimar una proporción o una media.
4. Aplicar la corrección por población finita.
5. Calcular una muestra para detectar diferencias entre grupos.
6. Ajustar el resultado a las condiciones reales de recogida.
7. Reconocer cuándo conviene usar una simulación, una curva de aprendizaje o un censo.

## PREPARACIÓN

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

## 1. Antes de calcular: ¿qué quieres conseguir?

El tamaño necesario depende del objetivo. Hay dos familias principales:

| Objetivo | Pregunta típica | Método principal |
|---|---|---|
| **Estimar con precisión** | ¿Qué porcentaje de clientes está satisfecho? ¿Cuál es el gasto medio? | Margen de error e intervalo de confianza |
| **Detectar un efecto** | ¿La versión B convierte mejor que la A? ¿Un tratamiento cambia la media? | Potencia estadística |

También existen métodos por **simulación**, útiles cuando el diseño es complejo, y **curvas de aprendizaje**, más propias de machine learning.

> **Idea clave:** una muestra adecuada para estimar una proporción con ±3 puntos porcentuales no es automáticamente adecuada para comparar dos grupos.

## 2. Los factores que determinan el tamaño

El tamaño no depende solamente de que la población tenga 10.000, 100.000 o un millón de personas. Depende sobre todo de:

- el **nivel de confianza**;
- el **margen de error** aceptable;
- la **variabilidad** de la característica estudiada;
- el **tamaño del efecto mínimo** que interesa detectar;
- la **potencia estadística** deseada;
- el diseño del muestreo;
- las pérdidas esperadas: no respuesta, abandono o datos inválidos.

Una población cien veces mayor no exige necesariamente una muestra cien veces mayor. En poblaciones grandes, la precisión depende mucho más del tamaño de la muestra que del tamaño total de la población.

## 3. Método 1: estimar una proporción

Se usa cuando el resultado es binario o se expresa como porcentaje: sí/no, compra/no compra, satisfecho/no satisfecho.

Para una población grande:

$$n_0 = \frac{z^2 p(1-p)}{e^2}$$

donde:

- $n_0$: tamaño inicial de la muestra;
- $z$: valor crítico asociado al nivel de confianza;
- $p$: proporción esperada, escrita entre 0 y 1;
- $e$: margen de error absoluto, escrito entre 0 y 1.

La fórmula del cuaderno original usa porcentajes: `50 × 50` y `me = 3`. Es equivalente a usar `0.5 × 0.5` y `e = 0.03`, pero **no se deben mezclar ambas escalas**.

In [ ]:
def muestra_proporcion(z=1.96, error=0.03, p=0.5):
    """Tamaño para estimar una proporción en una población grande."""
    if not 0 < error < 1:
        raise ValueError("error debe escribirse como proporción: 0.03 para un 3%")
    if not 0 < p < 1:
        raise ValueError("p debe estar entre 0 y 1")
    n = z**2 * p * (1 - p) / error**2
    return math.ceil(n)

muestra_proporcion(z=1.96, error=0.03, p=0.5)

El resultado es **1.068 personas**. Se redondea siempre hacia arriba, porque redondear hacia abajo reduciría la precisión prevista.

### 3.1. ¿Por qué se usa p = 0,5?

La cantidad $p(1-p)$ representa la variabilidad de una proporción. Alcanza su máximo cuando $p=0,5$. Por eso, si no sabemos nada de la proporción real, usar 50 % produce el tamaño más prudente.

Si un estudio piloto o una investigación anterior ofrece una estimación razonable, podemos usarla. Por ejemplo, si esperamos un 10 % de respuestas positivas, se necesita una muestra menor para el mismo margen de error.

In [ ]:
comparacion_p = pd.DataFrame({
    'proporción esperada': [0.50, 0.30, 0.10, 0.05],
})
comparacion_p['n necesario'] = comparacion_p['proporción esperada'].apply(
    lambda p: muestra_proporcion(1.96, 0.03, p)
)
comparacion_p

#### Punto de vigilancia

Elegir un valor de $p$ bajo solo para reducir el presupuesto no es correcto. Debe basarse en información previa defendible. Si existe mucha incertidumbre, se mantiene $p=0,5$.

### 3.2. Valores z habituales

| Nivel de confianza | z aproximado |
|---:|---:|
| 90 % | 1,645 |
| 95 % | 1,960 |
| 99 % | 2,576 |

El 95 % corresponde aproximadamente a **1,96** errores estándar, no exactamente a 2. El valor 2 produce un nivel cercano al 95,45 %. El 99 % corresponde a 2,576; tres errores estándar corresponden aproximadamente al 99,73 %.

In [ ]:
niveles = [0.90, 0.95, 0.99]
tabla_confianza = pd.DataFrame({'confianza': niveles})
tabla_confianza['z'] = tabla_confianza['confianza'].apply(
    lambda confianza: norm.ppf(1 - (1 - confianza) / 2)
)
tabla_confianza['n con error ±3%'] = tabla_confianza['z'].apply(
    lambda z: muestra_proporcion(z, 0.03, 0.5)
)
tabla_confianza.round(3)

### 3.3. Efecto del margen de error

El margen de error está elevado al cuadrado en el denominador. Por eso, reducir el error a la mitad exige aproximadamente **cuatro veces** más observaciones.

In [ ]:
errores = np.array([0.10, 0.07, 0.05, 0.04, 0.03, 0.02, 0.01])
tamaños = [muestra_proporcion(1.96, e, 0.5) for e in errores]

plt.figure(figsize=(8, 4))
plt.plot(errores * 100, tamaños, marker='o')
plt.gca().invert_xaxis()
plt.xlabel('Margen de error (± puntos porcentuales)')
plt.ylabel('Tamaño de muestra')
plt.title('Más precisión requiere una muestra mucho mayor')
plt.grid(alpha=0.3)
plt.show()

## 4. Método 2: corrección para una población finita

Cuando la población es pequeña y se muestrea **sin reemplazo**, se puede ajustar el tamaño inicial:

$$n = \frac{n_0}{1 + \frac{n_0-1}{N}}$$

donde $N$ es el tamaño de la población y $n_0$ el tamaño calculado como si la población fuera muy grande.

La corrección es especialmente útil cuando la muestra representa una fracción apreciable de la población, como orientación a partir del 5 % al 10 %. El umbral de 10.000 personas no es una frontera matemática: el efecto depende de la relación entre $n_0$ y $N$.

In [ ]:
def correccion_poblacion_finita(n0, poblacion):
    if poblacion <= 0:
        raise ValueError("La población debe ser positiva")
    n = n0 / (1 + (n0 - 1) / poblacion)
    return math.ceil(n)

n0 = muestra_proporcion(1.96, 0.03, 0.5)
poblaciones = [2_000, 5_000, 10_000, 100_000, 1_000_000]
pd.DataFrame({
    'población N': poblaciones,
    'muestra corregida': [correccion_poblacion_finita(n0, N) for N in poblaciones]
})

#### Ejemplo interpretado

Con 95 % de confianza, ±3 puntos porcentuales y $p=0,5$, obtenemos inicialmente 1.068 personas. Si la población completa tiene 10.000 personas, la corrección reduce la muestra a aproximadamente 965. Si tiene un millón, el resultado apenas cambia.

## 5. Método 3: estimar una media

Si queremos estimar una cantidad numérica —gasto medio, tiempo medio, puntuación media— necesitamos una estimación de la desviación estándar $\sigma$:

$$n_0 = \left(\frac{z\sigma}{e}\right)^2$$

Aquí $e$ es el error máximo aceptable en las **mismas unidades que la variable**. Si medimos tiempo en minutos, $\sigma$ y $e$ deben estar en minutos.

La desviación estándar puede proceder de un estudio anterior, datos históricos o una muestra piloto. Sin una estimación de la variabilidad no se puede calcular responsablemente este tamaño.

In [ ]:
def muestra_media(z, desviacion, error):
    if desviacion <= 0 or error <= 0:
        raise ValueError("La desviación y el error deben ser positivos")
    return math.ceil((z * desviacion / error) ** 2)

# Tiempo de atención: desviación estimada de 12 min; precisión deseada ±2 min
muestra_media(z=1.96, desviacion=12, error=2)

Necesitamos **139 observaciones** para estimar el tiempo medio con una precisión aproximada de ±2 minutos y un 95 % de confianza, suponiendo que la desviación estimada sea adecuada.

Si la población es pequeña, después se aplica la misma corrección por población finita del apartado anterior.

## 6. Método 4: estudio piloto

Un piloto no es una fórmula distinta, sino una forma de obtener los datos que faltan para usar una fórmula mejor.

- Para una proporción, permite estimar $p$.
- Para una media, permite estimar la desviación estándar.
- También permite estimar la tasa de respuesta, el abandono y la calidad del cuestionario.

El piloto debe parecerse al estudio definitivo. No conviene elegir su tamaño mediante una regla universal: depende del objetivo, del coste y de la variabilidad. Cuando la estimación piloto es inestable, se usa un escenario conservador y se documenta la incertidumbre.

> Los datos del piloto pueden incorporarse al estudio final únicamente si el protocolo, la población y la recogida no cambiaron y esta decisión estaba prevista.

## 7. Método 5: potencia estadística para detectar un efecto

Cuando queremos comparar grupos o probar una hipótesis, el cálculo debe basarse en la **potencia**, no solo en un margen de error.

Necesitamos definir antes de recoger los datos:

- $\alpha$: riesgo de falso positivo, a menudo 0,05;
- potencia $1-\beta$: probabilidad de detectar el efecto si existe, a menudo 0,80 o 0,90;
- efecto mínimo relevante: la diferencia más pequeña que justificaría una decisión;
- variabilidad y tipo de prueba;
- prueba unilateral o bilateral;
- reparto esperado entre grupos.

> **Muy importante:** el efecto mínimo relevante debe venir del problema científico, clínico o empresarial. No debe elegirse después de ver los datos.

### 7.1. Una media comparada con un valor de referencia

Para un test t de una muestra se suele expresar el efecto mediante $d$ de Cohen:

$$d = \frac{\mu_1-\mu_0}{\sigma}$$

Ejemplo: queremos detectar una diferencia de 3 puntos y esperamos una desviación estándar de 10. Entonces $d=0,3$.

In [ ]:
def muestra_potencia_una_media(diferencia, desviacion, alpha=0.05, potencia=0.80, bilateral=True):
    """Aproximación normal para una media o para diferencias pareadas."""
    z_alpha = norm.ppf(1 - alpha / 2) if bilateral else norm.ppf(1 - alpha)
    z_potencia = norm.ppf(potencia)
    n = ((z_alpha + z_potencia) * desviacion / diferencia) ** 2
    return math.ceil(n)

muestra_potencia_una_media(diferencia=3, desviacion=10)

### 7.2. Comparar dos medias independientes

Ejemplo: comparar el tiempo medio de dos procesos. Queremos detectar una diferencia mínima de 5 minutos y esperamos una desviación estándar común de 12 minutos. Calculamos $d=5/12$.

In [ ]:
def muestra_potencia_dos_medias(diferencia, desviacion, alpha=0.05, potencia=0.80, bilateral=True):
    """Aproximación normal; devuelve el tamaño de cada grupo de igual tamaño."""
    z_alpha = norm.ppf(1 - alpha / 2) if bilateral else norm.ppf(1 - alpha)
    z_potencia = norm.ppf(potencia)
    n_por_grupo = 2 * ((z_alpha + z_potencia) * desviacion / diferencia) ** 2
    return math.ceil(n_por_grupo)

n_por_grupo = muestra_potencia_dos_medias(diferencia=5, desviacion=12)
print('Personas por grupo:', n_por_grupo)
print('Total:', 2 * n_por_grupo)

La función supone que ambos grupos tienen el mismo tamaño. Si uno de los grupos será más pequeño, el total necesario suele aumentar. Son aproximaciones normales útiles para aprender y planificar; en un protocolo definitivo conviene usar una herramienta de potencia adaptada a la prueba exacta.

#### Diseño pareado

Si medimos a las mismas personas antes y después, las observaciones están emparejadas. El cálculo se basa en la desviación estándar de las **diferencias individuales**, no en la desviación de cada medición por separado. Si la correlación entre ambas mediciones es alta, el diseño pareado puede necesitar menos participantes.

### 7.3. Comparar dos proporciones: ejemplo A/B

Queremos detectar una subida de conversión del 20 % al 23 %. La diferencia absoluta es de 3 puntos porcentuales, pero el cálculo utiliza un tamaño del efecto específico para dos proporciones.

In [ ]:
def muestra_potencia_dos_proporciones(p1, p2, alpha=0.05, potencia=0.80, bilateral=True):
    """Aproximación normal; devuelve el tamaño de cada grupo de igual tamaño."""
    z_alpha = norm.ppf(1 - alpha / 2) if bilateral else norm.ppf(1 - alpha)
    z_potencia = norm.ppf(potencia)
    p_media = (p1 + p2) / 2
    numerador = (
        z_alpha * math.sqrt(2 * p_media * (1 - p_media))
        + z_potencia * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
    ) ** 2
    return math.ceil(numerador / (p2 - p1) ** 2)

p_control = 0.20
p_variante = 0.23
n_por_grupo_ab = muestra_potencia_dos_proporciones(p_control, p_variante)
print('Visitantes por grupo:', n_por_grupo_ab)
print('Total aproximado:', 2 * n_por_grupo_ab)

#### Una o dos colas

Una prueba unilateral puede necesitar menos muestra, pero solo es correcta si una diferencia en la dirección contraria no llevaría a rechazar la hipótesis nula y esa decisión se tomó antes del análisis. En la mayoría de estudios exploratorios, una prueba bilateral es la opción más prudente.

## 8. Método 6: simulación

Las fórmulas cerradas son cómodas, pero no cubren todos los diseños: medidas repetidas, varios niveles jerárquicos, métricas muy asimétricas, reglas de parada o modelos complejos. En esos casos se puede estimar la potencia por simulación:

1. Proponer un tamaño $n$.
2. Generar muchos estudios artificiales bajo un efecto considerado realista.
3. Aplicar exactamente el análisis previsto.
4. Calcular en qué proporción de simulaciones se detecta el efecto.
5. Repetir con otros tamaños hasta alcanzar la potencia deseada.

La simulación no elimina las hipótesis: las hace explícitas. Su resultado será tan razonable como el modelo utilizado para generar los datos.

In [ ]:
from scipy.stats import ttest_ind

def potencia_simulada_dos_medias(n_por_grupo, diferencia=5, desviacion=12,
                                 repeticiones=2000, alpha=0.05, semilla=1234):
    rng = np.random.default_rng(semilla)
    rechazos = 0
    for _ in range(repeticiones):
        grupo_a = rng.normal(0, desviacion, n_por_grupo)
        grupo_b = rng.normal(diferencia, desviacion, n_por_grupo)
        pvalor = ttest_ind(grupo_a, grupo_b, equal_var=False).pvalue
        rechazos += pvalor < alpha
    return rechazos / repeticiones

for n in [40, 60, 80, 100, 120]:
    print(n, round(potencia_simulada_dos_medias(n), 3))

Como es una simulación, el resultado cambia ligeramente. Aumentar el número de repeticiones mejora su estabilidad, pero también aumenta el tiempo de cálculo.

## 9. Método 7: curvas de aprendizaje en machine learning

En machine learning, el tamaño útil también puede estudiarse empíricamente:

1. entrenar el modelo con fracciones crecientes de los datos;
2. medir el rendimiento mediante validación;
3. representar rendimiento frente a tamaño de entrenamiento;
4. observar si añadir datos sigue mejorando el modelo.

Esto ayuda a decidir si recoger más datos probablemente aportará valor. No sustituye al cálculo de potencia cuando la pregunta es inferencial, y la evaluación debe hacerse con un conjunto de validación o test independiente.

Además del número de filas importan la frecuencia de las clases raras, la calidad de las etiquetas, el número de variables y la diversidad de casos.

## 10. Ajustes necesarios en un estudio real

### 10.1. No respuesta, abandono y datos inválidos

La fórmula produce el número de observaciones **analizables**, no necesariamente el número de personas a contactar.

Si esperamos una tasa de respuesta $r$:

$$n_{contactar} = \frac{n_{analizable}}{r}$$

Si esperamos una pérdida $a$, la tasa conservada es $1-a$:

$$n_{reclutar} = \frac{n_{analizable}}{1-a}$$

No se debe calcular simplemente $n(1+a)$: dividir por la tasa conservada garantiza que el tamaño final esperado sea suficiente.

In [ ]:
def ajustar_perdidas(n_analizable, tasa_conservada):
    if not 0 < tasa_conservada <= 1:
        raise ValueError("La tasa debe estar entre 0 y 1")
    return math.ceil(n_analizable / tasa_conservada)

n_analizable = 1068
print('Con 80% de respuesta, contactar:', ajustar_perdidas(n_analizable, 0.80))
print('Con 15% de abandono, reclutar:', ajustar_perdidas(n_analizable, 0.85))

### 10.2. Muestreo por conglomerados y efecto de diseño

Si seleccionamos grupos completos —aulas, hospitales, municipios—, las personas del mismo grupo suelen parecerse. Aportan menos información independiente que el mismo número de personas seleccionadas al azar.

Una aproximación es multiplicar por el **efecto de diseño** $DEFF$:

$$n_{diseño} = n_{aleatorio} \times DEFF$$

Cuando los conglomerados tienen un tamaño medio $m$ y una correlación intraclase $ICC$:

$$DEFF \approx 1 + (m-1)ICC$$

Ejemplo: con aulas de 20 estudiantes e $ICC=0,05$, $DEFF=1,95$. Haría falta casi el doble de muestra que con un muestreo aleatorio simple.

In [ ]:
def efecto_diseño(tamaño_medio_conglomerado, correlacion_intraclase):
    return 1 + (tamaño_medio_conglomerado - 1) * correlacion_intraclase

deff = efecto_diseño(20, 0.05)
n_ajustado = math.ceil(1068 * deff)
deff, n_ajustado

### 10.3. Estratos y subgrupos importantes

El muestreo estratificado puede mejorar la precisión si los estratos son internamente homogéneos. Sin embargo, el tamaño total no basta si queremos resultados precisos por región, edad o sexo.

Si necesitamos una conclusión independiente para cada subgrupo, debemos comprobar el tamaño dentro de cada uno. A veces se sobrerrepresenta un grupo pequeño y después se aplican ponderaciones para estimar resultados poblacionales.

Las ponderaciones pueden recuperar la representatividad de las estimaciones, pero suelen aumentar su varianza. El cálculo definitivo debe tener en cuenta el diseño completo.

### 10.4. Análisis múltiples

Probar muchas hipótesis aumenta la probabilidad de falsos positivos. Si existe un resultado principal, el tamaño se calcula para ese resultado. Si hay varios resultados principales o muchas comparaciones, puede ser necesario ajustar $\alpha$ y aumentar la muestra.

Conviene definir de antemano qué análisis es principal y cuáles son exploratorios.

## 11. Representatividad: lo que una muestra grande no arregla

Las fórmulas cuantifican principalmente el **error aleatorio**, suponiendo que el diseño es adecuado. No corrigen automáticamente:

- sesgo de selección;
- ausencia de grupos importantes;
- baja cobertura del marco muestral;
- preguntas ambiguas o mediciones defectuosas;
- no respuesta relacionada con la variable estudiada;
- datos duplicados o etiquetas erróneas.

Una muestra enorme de voluntarios puede ser menos representativa que una muestra aleatoria mucho menor. **Tamaño y representatividad son problemas distintos.**

## 12. ¿Cuándo hacer un censo?

A veces conviene observar a toda la población:

- la población es pequeña y accesible;
- cada caso es importante para una decisión administrativa;
- los datos ya existen y procesarlos tiene un coste razonable;
- se necesitan resultados detallados para subgrupos muy pequeños.

Un censo elimina el error de muestreo para esa población y ese momento, pero no elimina errores de medición, valores ausentes, problemas de cobertura ni la incertidumbre sobre el futuro.

## 13. Reglas rápidas: útiles solo para orientarse

Afirmaciones como «30 observaciones son suficientes», «10 casos por variable» o «usa el 10 % de la población» no son métodos universales. Pueden servir como primera orientación o control de plausibilidad, pero no sustituyen un cálculo ligado al objetivo, la variabilidad, el diseño y el análisis previsto.

En particular, 30 no garantiza normalidad, potencia ni representatividad.

## 14. Guía de decisión

| Situación | Método recomendado | Información necesaria |
|---|---|---|
| Estimar un porcentaje | Fórmula de una proporción | Confianza, margen de error, $p$ esperada |
| Estimar una media | Fórmula de una media | Confianza, margen de error, desviación estándar |
| Población pequeña | Corrección por población finita | $N$ y tamaño inicial $n_0$ |
| Comparar grupos o detectar un cambio | Análisis de potencia | $\alpha$, potencia, efecto mínimo, variabilidad |
| Diseño complejo | Simulación | Modelo de datos y análisis previsto |
| Decidir si más datos ayudan a un modelo predictivo | Curva de aprendizaje | Rendimiento validado con tamaños crecientes |
| Población pequeña y accesible | Censo | Coste, cobertura y calidad posibles |

Después del cálculo, aplica en este orden los ajustes relevantes: **diseño**, **subgrupos** y **pérdidas esperadas**.

## 15. Función práctica para estimaciones

La función siguiente reúne los casos de estimación de una proporción y de una media. El análisis de potencia queda separado porque necesita parámetros propios del contraste previsto.

In [ ]:
def calcular_muestra_estimacion(tipo, confianza=0.95, error=None,
                                p=0.5, desviacion=None, poblacion=None,
                                efecto_diseño=1.0, tasa_conservada=1.0):
    if not 0 < confianza < 1:
        raise ValueError("confianza debe estar entre 0 y 1")
    if error is None or error <= 0:
        raise ValueError("Debes indicar un error positivo")
    if efecto_diseño < 1:
        raise ValueError("El efecto de diseño debe ser al menos 1 en esta función")
    if not 0 < tasa_conservada <= 1:
        raise ValueError("tasa_conservada debe estar entre 0 y 1")

    z = norm.ppf(1 - (1 - confianza) / 2)

    if tipo == 'proporcion':
        n0 = z**2 * p * (1 - p) / error**2
    elif tipo == 'media':
        if desviacion is None or desviacion <= 0:
            raise ValueError("Para una media debes indicar una desviación positiva")
        n0 = (z * desviacion / error) ** 2
    else:
        raise ValueError("tipo debe ser 'proporcion' o 'media'")

    n_corregida = n0 if poblacion is None else n0 / (1 + (n0 - 1) / poblacion)
    n_diseño = n_corregida * efecto_diseño
    n_reclutar = n_diseño / tasa_conservada

    return {
        'z': round(z, 3),
        'n inicial': math.ceil(n0),
        'n tras población finita': math.ceil(n_corregida),
        'n analizable tras diseño': math.ceil(n_diseño),
        'n a reclutar o contactar': math.ceil(n_reclutar)
    }

In [ ]:
# Ejemplo completo:
# proporción, 95% de confianza, ±3 puntos, población de 10.000,
# efecto de diseño 1,2 y 80% de respuestas válidas.
calcular_muestra_estimacion(
    tipo='proporcion', confianza=0.95, error=0.03, p=0.5,
    poblacion=10_000, efecto_diseño=1.2, tasa_conservada=0.80
)

## 16. Ejercicios

### Ejercicio 1 — Encuesta
Una ciudad quiere estimar el porcentaje de habitantes que utiliza un servicio. Desea 95 % de confianza y ±4 puntos porcentuales. No existe información previa sobre la proporción.

1. ¿Qué valor de $p$ debes usar?
2. ¿Qué tamaño necesitas si la población es grande?
3. ¿Qué cambia si la ciudad solo tiene 3.000 habitantes?

### Ejercicio 2 — Tiempo medio
Un piloto estima una desviación estándar de 18 minutos. Se quiere estimar el tiempo medio con 95 % de confianza y ±3 minutos. Calcula la muestra necesaria.

### Ejercicio 3 — Pérdidas
Un análisis necesita 400 casos válidos y se espera conservar el 70 % de las respuestas. ¿A cuántas personas hay que contactar?

### Ejercicio 4 — Elegir el método
Quieres comprobar si una nueva página aumenta la conversión del 12 % al 14 %. ¿Usarías la fórmula de margen de error o un análisis de potencia? Explica por qué.

## 17. Soluciones

### Solución 1
Se usa $p=0,5$ porque no existe información previa. Para población grande:

```python
n0 = muestra_proporcion(1.96, 0.04, 0.5)
n_finita = correccion_poblacion_finita(n0, 3000)
n0, n_finita
```

Se obtienen 601 personas para población grande y aproximadamente 501 después de la corrección finita.

### Solución 2

```python
muestra_media(1.96, 18, 3)
```

Se necesitan 139 observaciones.

### Solución 3

$400 / 0,70 = 571,43$. Hay que contactar al menos a **572 personas**.

### Solución 4

Se usa un análisis de potencia para comparar dos proporciones, porque la pregunta es si podemos **detectar una diferencia** entre A y B. La fórmula de margen de error responde a otra pregunta: con qué precisión estimamos una sola proporción.

## 18. Checklist final

Antes de aprobar un tamaño de muestra, comprueba:

1. ¿La pregunta principal es estimar o comparar?
2. ¿La variable principal es una proporción, una media u otra métrica?
3. ¿El margen de error o el efecto mínimo tiene sentido práctico?
4. ¿El nivel de confianza, $\alpha$ y la potencia se fijaron antes de analizar?
5. ¿La estimación de $p$ o de la desviación tiene una justificación?
6. ¿Hace falta corrección por población finita?
7. ¿Hay conglomerados, estratos o subgrupos que analizar?
8. ¿Se ajustó la no respuesta, el abandono y los datos inválidos?
9. ¿El procedimiento de selección evita sesgos importantes?
10. ¿Se documentaron las hipótesis y se redondeó hacia arriba?

## CONCLUSIÓN

No existe un número mágico ni una fórmula válida para todos los estudios.

- Para **estimar**, elegimos la precisión y el nivel de confianza.
- Para **comparar o detectar**, elegimos la potencia y el efecto mínimo relevante.
- Para diseños complejos, podemos usar una **simulación**.
- En machine learning, una **curva de aprendizaje** ayuda a valorar si más datos mejorarán el modelo.
- Después ajustamos población finita, diseño, subgrupos y pérdidas.

La muestra debe ser suficientemente grande, pero también debe estar bien seleccionada y bien medida. Una muestra mayor reduce el error aleatorio; no elimina por sí sola los sesgos.